# ITOM-6229 Lab: CoT vs. Act-only vs. ReAct

Read [LAB.md](LAB.md). Use the same model and question IDs for all methods. CoT has no tools; Act-only and ReAct share the same tools and step limit. This is a teaching comparison, not a reproduction of the paper. Run from the repository root.

Set `OPENAI_API_KEY` as a Codespaces secret. Set `OPENAI_MODEL` to the instructor-approved model ID below.


In [4]:
import os
import requests
from pathlib import Path
from openai import OpenAI

# Local runs: load KEY=value lines from .env without overriding variables already set (e.g. Codespaces secrets).
if Path(".env").exists():
    for line in Path(".env").read_text().splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "")  # Enter the instructor-approved model ID if needed.
if not OPENAI_MODEL:
    raise ValueError("Set OPENAI_MODEL to the instructor-approved Responses API model ID.")
if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError("Add OPENAI_API_KEY as a Codespaces secret and restart the codespace.")
client = OpenAI(timeout=60.0, max_retries=0)

# Shared model and output budget. No temperature is supplied because model support varies.
MAX_OUTPUT_TOKENS = 1024

def generate(prompt, instructions):
    response = client.responses.create(
        model=OPENAI_MODEL, instructions=instructions, input=prompt,
        max_output_tokens=MAX_OUTPUT_TOKENS, store=False,
    )
    if not response.output_text.strip():
        raise RuntimeError("Empty output; check model access and the output-token budget.")
    return response.output_text.strip()

In [7]:
import wikienv, wrappers
env = wikienv.WikiEnv()
env = wrappers.HotPotQAWrapper(env, split="dev")
env = wrappers.LoggingWrapper(env)

def step(env, action):
    attempts = 0
    while attempts < 3:
        try:
            return env.step(action)
        except requests.exceptions.Timeout:
            attempts += 1
    raise TimeoutError("Wikipedia timed out three times. Record the failure and retry later.")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## Three methods, matched examples

The original prompt file supplies `cotqa_simple6`, `webact_simple6`, and `webthink_simple6`. Check that they use the same six demonstration questions. Act-only omits explicit reasoning text; it does not disable a model's internal reasoning.


In [8]:
import json
import re
import time
from pathlib import Path
from wrappers import normalize_answer

with open("prompts/prompts_naive.json") as f:
    prompt_sets = json.load(f)
PROMPT_KEYS = {"cot": "cotqa_simple6", "act": "webact_simple6", "react": "webthink_simple6"}
examples = {method: prompt_sets[key] for method, key in PROMPT_KEYS.items()}
example_questions = [re.findall(r"^Question: (.+)$", text, re.MULTILINE) for text in examples.values()]
assert len(example_questions[0]) == 6 and all(q == example_questions[0] for q in example_questions)
MAX_STEPS = 7
TOOLS = (
    "Search[entity] retrieves a Wikipedia page or similar titles; "
    "Lookup[keyword] retrieves the next matching sentence in the current page; "
    "Finish[answer] ends the task with a short answer. "
)

def run_method(method, idx):
    if method not in PROMPT_KEYS:
        raise ValueError("Unknown method")
    question = env.reset(idx=idx)  # Fresh environment for each method.
    history = examples[method] + "\n" + question + "\n"
    record = dict(question_id=idx, question=question, method=method, model=OPENAI_MODEL,
                  max_output_tokens=MAX_OUTPUT_TOKENS, max_steps=MAX_STEPS if method != "cot" else 0,
                  answer=None, em=None, status="running", model_calls=0, tool_calls=0,
                  invalid_format_calls=0, steps=[], trajectory="", elapsed_seconds=0)
    start = time.monotonic()
    try:
        if method == "cot":
            record["model_calls"] += 1
            output = generate(history, "Answer without tools. Provide a brief explanation, then a final line 'Answer: <short answer>'.")
            history += output + "\n"
            match = re.search(r"^Answer:\s*(.+)$", output, re.MULTILINE)
            if not match:
                record["invalid_format_calls"] += 1
                record["status"] = "format_error"
            else:
                record["answer"] = match.group(1).strip()
                record["status"] = "finished"
        else:
            for i in range(1, MAX_STEPS + 1):
                prefix = f"Action {i}:" if method == "act" else f"Thought {i}:"
                instructions = TOOLS + "Continue the supplied text at its end. Emit only the next step. Never invent observations. "
                instructions += ("Emit only the action argument after the supplied Action prefix; no Thought lines." if method == "act" else
                                 f"Emit a brief action rationale after the Thought prefix, then 'Action {i}: <action>'.")
                record["model_calls"] += 1
                output = generate(history + prefix, instructions)
                # Retain actual output to make format failures inspectable.
                output = output.split(f"\nObservation {i}:", 1)[0].strip()
                if method == "react":
                    match = re.search(rf"(?:^|\n)Action {i}:\s*(.+)", output)
                    action = match.group(1).strip() if match else ""
                else:
                    action = re.sub(rf"^Action {i}:\s*", "", output).strip()
                valid = re.fullmatch(r"(Search|Lookup|Finish)\[(.*?)\]", action)
                if not valid:
                    record["invalid_format_calls"] += 1
                    obs = "Invalid format. Use exactly one Search[entity], Lookup[keyword], or Finish[answer] action."
                    done = False
                else:
                    normalized = action[0].lower() + action[1:]
                    if valid.group(1) in ("Search", "Lookup"):
                        record["tool_calls"] += 1
                    obs, reward, done, info = step(env, normalized)
                    if done:
                        record["answer"] = info["answer"]
                        record["status"] = "finished"
                record["steps"].append(dict(step=i, model_output=output, action=action, observation=obs))
                history += prefix + output + f"\nObservation {i}: {obs}\n"
                if done:
                    break
            if record["status"] == "running":
                record["status"] = "step_limit"
        # Score completed or exhausted attempts; infrastructure/format failures stay unscored.
        ground_truth = env.env.data[idx][1]
        record["ground_truth"] = ground_truth
        if record["status"] in ("finished", "step_limit"):
            record["em"] = int(normalize_answer(record["answer"] or "") == normalize_answer(ground_truth))
    except Exception as error:
        # Preserve completed runs and identify operational failures separately from wrong answers.
        record["status"] = "execution_error"
        record["error_type"] = type(error).__name__
        print(method, idx, "stopped:", type(error).__name__, "— check access/network before retrying.")
    finally:
        record["trajectory"] = history
        record["elapsed_seconds"] = round(time.monotonic() - start, 2)
    return record


In [9]:
QUESTION_IDS = [0, 1, 2]
METHODS = ["cot", "act", "react"]
results = []
Path("results").mkdir(exist_ok=True)
for idx in QUESTION_IDS:
    for method in METHODS:
        record = run_method(method, idx)
        results.append(record)
        Path("results/hotpotqa_results.json").write_text(json.dumps(results, indent=2))
        print(idx, method, record["status"], "answer:", record["answer"], "EM:", record["em"])
        if record["status"] == "execution_error":
            raise RuntimeError("Run stopped after an execution error. Partial results were saved; diagnose before rerunning.")
print("Saved", len(results), "method/question runs.")


0 cot finished answer: Yes EM: 1
0 act finished answer: Yes, both Scott Derrickson and Ed Wood are American filmmakers. EM: 0
0 react finished answer: yes EM: 1
1 cot finished answer: U.S. Ambassador EM: 0
1 act step_limit answer: None EM: 0
1 react step_limit answer: None EM: 0
2 cot finished answer: Across the Universe series EM: 0
2 act finished answer: Animorphs EM: 1
2 react finished answer: Animorphs EM: 1
Saved 9 method/question runs.


## Compare outcomes and inspect traces

There should be nine records: three questions × three methods. An infrastructure error has `em=null` and is not an incorrect answer. Act/ReAct stopping at the step limit without an answer counts as an unsuccessful attempt. Compare prompt examples, answers, model calls, tool calls, runtime, and action–observation sequences.


In [10]:
print("ID | Method | Answer | EM | Model calls | Tool calls | Seconds | Status")
for r in results:
    print(r["question_id"], r["method"], r["answer"], r["em"], r["model_calls"],
          r["tool_calls"], r["elapsed_seconds"], r["status"], sep=" | ")

# Inspect the first question's three traces; change the ID to explore another.
for r in results:
    if r["question_id"] == QUESTION_IDS[0]:
        print("\nMETHOD:", r["method"], "\n", r["trajectory"])


ID | Method | Answer | EM | Model calls | Tool calls | Seconds | Status
0 | cot | Yes | 1 | 1 | 0 | 2.67 | finished
0 | act | Yes, both Scott Derrickson and Ed Wood are American filmmakers. | 0 | 5 | 4 | 7.26 | finished
0 | react | yes | 1 | 3 | 2 | 4.34 | finished
1 | cot | U.S. Ambassador | 0 | 1 | 0 | 1.0 | finished
1 | act | None | 0 | 7 | 7 | 13.61 | step_limit
1 | react | None | 0 | 7 | 7 | 15.12 | step_limit
2 | cot | Across the Universe series | 0 | 1 | 0 | 1.16 | finished
2 | act | Animorphs | 1 | 5 | 4 | 7.86 | finished
2 | react | Animorphs | 1 | 5 | 4 | 10.12 | finished

METHOD: cot 
 Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?
Thought: Let's think step by step. The eastern sector of Colorado orogeny extends into the High Plains. High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.
Answer: 1,800 to 7,000 ft
Question: Musician and satirist Allie Goertz wrote a son

## Your analysis

Add your name and observations from Parts 1–5 here, and complete `LAB_NOTES.md`.
